In [ ]:
import torch
import json
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.models import StrongCNN
from src.dataset import get_dataloaders
from src.utils import get_device, set_seed

set_seed(42)
device = get_device()

# Load saved weights
model = StrongCNN()
model.load_state_dict(torch.load('../checkpoints/strong_cnn_best.pth',
                                  map_location=device))
model.to(device)
model.eval()

# Run on test set
_, _, test_loader = get_dataloaders(batch_size=32, augment=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

# Classification report
classes = ['AnnualCrop','Forest','HerbaceousVegetation','Highway',
           'Industrial','Pasture','PermanentCrop','Residential','River','SeaLake']

report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=classes))

# Save results
with open('../results/strong_cnn_results.json', 'w') as f:
    json.dump(report, f, indent=2)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.title('StrongCNN Confusion Matrix')
plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=150)
plt.show()